# Notebook 06: Bilingual RAG Architecture

## Overview

This notebook provides an end-to-end walkthrough showing how the same question is handled in Arabic vs English, and summarizes the architecture decisions that make bilingual RAG work.

```
Arabic Question  -->  Normalize  -->  Intent  -->  Retrieve  -->  Generate (Arabic)

English Question -->  Normalize  -->  Intent  -->  Retrieve  -->  Generate (English)
```

**What you will see:**
- Same question in Arabic and English, processed side by side
- How normalization handles Arabic variants
- How retrieval results compare across languages
- How generation adapts to the question language
- Architecture summary with all design decisions

## 1. Setup

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.rag_app import config
from src.rag_app.retrieval.query_understanding import (
    normalize_question,
    understand_question,
    add_domain_context,
    INTENTS,
)
from src.rag_app.retrieval.search import search, get_embedding_model
from src.rag_app.utils.text import is_arabic
from src.rag_app.generation.citation import citation_for

In [ ]:
model = get_embedding_model()

## 2. Same Question, Two Languages

We use a follow-up question that a patient might ask in either language.

In [ ]:
arabic_q = "ايه المتابعة المطلوبة بعد الجراحة؟"
english_q = "What follow-up is needed after surgery?"

print(f"Arabic:   '{arabic_q}'  --> is_arabic: {is_arabic(arabic_q)}")
print(f"English:  '{english_q}'  --> is_arabic: {is_arabic(english_q)}")

## 3. Normalization Comparison

In [ ]:
print("Arabic normalization:")
print(f"  Original:  '{arabic_q}'")
print(f"  Normalized: '{normalize_question(arabic_q)}'")

print("\nEnglish normalization:")
print(f"  Original:  '{english_q}'")
print(f"  Normalized: '{normalize_question(english_q)}'")

## 4. Intent Classification Comparison

In [ ]:
arabic_result = understand_question(arabic_q, model)
english_result = understand_question(english_q, model)

print(f"Arabic intent:  {arabic_result['intent']} (confidence: {arabic_result['confidence']:.2f})")
print(f"English intent: {english_result['intent']} (confidence: {english_result['confidence']:.2f})")

print(f"\nArabic queries ({len(arabic_result['queries'])}):")
for i, q in enumerate(arabic_result['queries'], 1):
    print(f"  {i}. {q}")

print(f"\nEnglish queries ({len(english_result['queries'])}):")
for i, q in enumerate(english_result['queries'], 1):
    print(f"  {i}. {q}")

## 5. Domain Context Expansion

In [ ]:
print("Arabic domain expansion:")
print(f"  Original: '{arabic_q}'")
print(f"  Expanded: '{add_domain_context(arabic_q)}'")

print("\nEnglish domain expansion:")
print(f"  Original: '{english_q}'")
print(f"  Expanded: '{add_domain_context(english_q)}'")

## 6. Retrieval Comparison

In [ ]:
arabic_rows = search(arabic_q)
english_rows = search(english_q)

print("Arabic retrieval:")
for row in arabic_rows:
    print(f"  #{row['rank']} (score: {row['rerank_score']:.4f}) Page {row['page_number']}: {row['section_title']}")

print("\nEnglish retrieval:")
for row in english_rows:
    print(f"  #{row['rank']} (score: {row['rerank_score']:.4f}) Page {row['page_number']}: {row['section_title']}")

## 7. Citation Comparison

In [ ]:
top_arabic = arabic_rows[0]
top_english = english_rows[0]

print("Arabic citation:")
print(f"  {citation_for(top_arabic, True)}")

print("\nEnglish citation:")
print(f"  {citation_for(top_english, False)}")

## 8. Why Bilingual RAG Needs Special Handling

### The Challenge

1. **Arabic text has many valid spellings** - "أعراض" and "اعراض" are the same word
2. **Short questions are ambiguous** - "ايه العلاج؟" (what's the treatment?) needs context
3. **Evidence is in English** - NICE guidelines are English, but users ask in Arabic
4. **Citations must be bilingual** - Arabic users need Arabic section names

### Our Solutions

| Challenge | Solution |
|-----------|----------|
| Arabic spelling variants | NFC normalization + alef/ya/diacritics normalization |
| Ambiguous short questions | Domain context expansion ("في سياق سرطان القولون والمستقيم:") |
| Cross-language retrieval | Multilingual E5 embeddings (shared vector space) |
| Cross-language claim check | E5 semantic similarity (threshold 0.72) instead of lexical overlap |
| Bilingual citations | Arabic section name translation map (15 entries) |
| Bilingual output | Language-specific output format templates (Recommendation/Excerpt/Citation) |

## 9. Architecture Summary

```
                          INGESTION
                          =========
NICE NG151 PDF  -->  PDF Loader  -->  Markdown Cleaning  -->  Structure-Aware Chunking
NICE NG12 PDF   -->  (supplementary colorectal section)         |
                                                                v
                                                    Token Count Validation
                                                    (CHUNK_SIZE = 450)
                                                                |
                                                                v
                                                    Embedding (multilingual-e5-base)
                                                                |
                                                                v
                                                    Chroma (cosine HNSW)

                          RETRIEVAL
                          =========
User Question  -->  Query Understanding
                     |  - Normalize (Arabic/English)
                     |  - Classify intent (cue match or semantic)
                     |  - Expand domain context
                     |  - Generate multiple queries
                     v
              Multi-Query Encoding
                     |
                     v
              Chroma Search (top_k * 4 candidates)
                     |
                     v
              Deduplication + Reranking
              (semantic + lexical + intent boost)
                     |
                     v
              Top-K Results

                          GENERATION
                          ==========
Top-K Results  -->  Build Context (numbered PASSAGEs with citations)
                         |
                         v
                  System Prompt (citation-bound, calibrated, concise)
                         |
                         v
                  LLM Call (Groq: qwen3.6-27b, temp=0.2)
                         |
                         v
                  Citation Validation
                         |
                         v
                  Claim Support Check
                  (English: lexical, Arabic: E5 semantic)
                         |
                         v
                  Grounded Answer + Disclaimer

                          EVALUATION
                          ==========
66 Test Questions  -->  Batch Retrieve  -->  Relevance Matching
                                               |
                                               v
                                        Metrics (Found Rate, MAP@k, MRR)
```

### Key Configuration

| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Embedding model | multilingual-e5-base | Best bilingual performance |
| Chunk size | 450 tokens | Under E5's 512 limit |
| Chunk overlap | 80 tokens | Preserves context across chunks |
| Top-k | 5 | Best recall/precision balance |
| Generation model | qwen3.6-27b (Groq) | Free, fast, good bilingual |
| Temperature | 0.2 | Deterministic for medical answers |
| MIN_RETRIEVAL_SCORE | 0.75 | Refuse when evidence is weak |

### Performance (Current Configuration)

| Metric | Value |
|--------|-------|
| Found rate (k=5) | 90.6% |
| MAP@5 | 67.0% |
| MRR | 69.0% |
| AR found rate | ~90% |
| EN found rate | ~91% |